# Big Data Analytics (BDA) & Spatial Gap Analysis — Surabaya Timur
### Pemetaan Geospasial, Autokorelasi Spasial, dan Analisis Kesenjangan Digitalisasi

**Identitas Mahasiswa & Kelompok:**
- **NIM:** 09040624091
- **Nama:** Muhammad Iqbal Rizanta
- **Kelompok:** 3
- **Mata Kuliah:** Big Data Analytics
- **Program Studi:** Sistem Informasi, Fakultas Sains dan Teknologi, UINSA
- **Dosen Pengampu:** Subhan Nooriansyah, M.Kom.

---
## 1. Pendahuluan Analisis BDA & Spasial
Tahap BDA menyatukan hasil ekstraksi dan modeling ke dalam perspektif makro geospasial untuk menjawab kebutuhan pemangku kebijakan:
1. **Agregasi Big Data:** Menghitung indeks adopsi digital per kecamatan di Surabaya Timur.
2. **Pemetaan Spasial:** Visualisasi sebaran UMKM dan klaster sentra kuliner (SWK Convention Hall & SWK Deles).
3. **Gap Analysis:** Mengidentifikasi kecamatan prioritas yang memerlukan intervensi insentif / pendampingan pembayaran digital.


In [ ]:
# 1. Import Pustaka BDA & Geospasial
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

clean_dir = Path('../data/clean') if Path('../data/clean').exists() else Path('data/clean')
df_umkm = pd.read_parquet(clean_dir / 'umkm.parquet')
df_ulasan = pd.read_parquet(clean_dir / 'ulasan.parquet')
print('Data spasial UMKM berhasil dimuat!')


---
## 2. Agregasi Spasial per Kecamatan di Surabaya Timur
Menghitung persentase adopsi pembayaran digital (QRIS, POS, EDC, Merchant Online) per kecamatan.


In [ ]:
# 2.1 Agregasi Sinyal Digital per Kecamatan
bda_agg = df_ulasan.groupby('kecamatan').agg({
    'mention_qris': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_pos': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_edc': lambda x: round((x.sum() / len(x)) * 100, 2),
    'mention_online': lambda x: round((x.sum() / len(x)) * 100, 2),
    'rating_pengulas': 'mean',
    'ulasan_id': 'count'
}).reset_index()

bda_agg.columns = ['Kecamatan', 'Adopsi QRIS (%)', 'Adopsi POS (%)', 'Adopsi EDC (%)', 'Adopsi Ojol (%)', 'Rata-rata Rating', 'Total Ulasan']
display(bda_agg)


In [ ]:
# 2.2 Visualisasi Perbandingan Adopsi Digital Antar-Kecamatan
plt.figure(figsize=(10, 5))
bar_data = bda_agg.set_index('Kecamatan')[['Adopsi QRIS (%)', 'Adopsi POS (%)', 'Adopsi EDC (%)', 'Adopsi Ojol (%)']]
bar_data.plot(kind='bar', figsize=(11, 5), colormap='viridis')
plt.title('Tingkat Penetrasi Indikator Digitalisasi UMKM Kuliner per Kecamatan di Surabaya Timur', fontsize=13, fontweight='bold')
plt.ylabel('Persentase Penetrasi (%)', fontsize=11)
plt.xlabel('Kecamatan', fontsize=11)
plt.xticks(rotation=15)
plt.legend(title='Indikator')
plt.tight_layout()
plt.show()


---
## 3. Analisis Kesenjangan (Gap Analysis) & Rekomendasi Kebijakan
Menganalisis kesenjangan adopsi digital antar-wilayah untuk merumuskan rekomendasi intervensi Dinas Koperasi & UKM.


In [ ]:
# 3.1 Perhitungan Indeks Kesenjangan (Digital Gap Index)
# Dihitung berdasarkan selisih rata-rata penetrasi relatif terhadap kecamatan terdepan
bda_agg['Skor Kematangan Rata-Rata'] = (
    bda_agg['Adopsi QRIS (%)'] * 0.35 +
    bda_agg['Adopsi POS (%)'] * 0.25 +
    bda_agg['Adopsi EDC (%)'] * 0.15 +
    bda_agg['Adopsi Ojol (%)'] * 0.25
)

bda_agg['Prioritas Intervensi'] = bda_agg['Skor Kematangan Rata-Rata'].apply(
    lambda s: 'Prioritas Tinggi (Butuh Pendampingan)' if s < 10.0 else (
        'Prioritas Sedang (Perluasan POS/EDC)' if s < 20.0 else 'Mandiri / Maju'
    )
)

display(bda_agg[['Kecamatan', 'Skor Kematangan Rata-Rata', 'Prioritas Intervensi']].sort_values(by='Skor Kematangan Rata-Rata'))


---
## 4. Analisis Khusus Sentra Wisata Kuliner (SWK Convention Hall & Deles)
Mengevaluasi sentra kuliner binaan di Kecamatan Sukolilo sebagai pusat percontohan digitalisasi UMKM.


In [ ]:
# 4.1 Filter Entitas SWK Sukolilo
swk_data = df_umkm[df_umkm['tipe_lokasi'].isin(['SWK', 'SWK_Stan_Survei'])]
print(f'Total entitas terkait Sentra Wisata Kuliner (SWK): {len(swk_data)}')
print(swk_data[['nama_umkm', 'tipe_lokasi', 'alamat']].head(5))


---
## 5. Rekomendasi Aksi untuk Pemangku Kepentingan
Berdasarkan analisis BDA spasial di Surabaya Timur:
1. **Fokus Intervensi QRIS:** Sosialisasi QRIS massal untuk kecamatan pinggiran/perbatasan (Gunung Anyar & Mulyorejo) untuk menggeser UMKM Konvensional menjadi Berkembang.
2. **Penyediaan POS Murah:** Sentra Wisata Kuliner (seperti SWK Convention Hall & Deles) memerlukan standardisasi sistem POS bersama agar pencatatan transaksi lebih akurat dan terintegrasi.
3. **Dukungan Kemitraan Ojol:** Fasilitasi kemitraan kolektif dengan platform online (GoFood/ShopeeFood/GrabFood) agar stan kuliner SWK dapat menjangkau konsumen lebih luas di luar jam makan malam.
